# 03 · Spark, Delta Lake, and data quality

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** databricks. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Build typed Spark data, quarantine invalid rows, use Delta MERGE, and inspect history.


## Step 1 — Choose an isolated namespace

Every Spark lab uses a schema derived from your user identity in the current catalog. Tables are course-owned. Set a writable catalog before running if needed.


In [ ]:
import hashlib
from pyspark.sql import functions as F
user = spark.sql('SELECT current_user()').first()[0]
catalog = spark.sql('SELECT current_catalog()').first()[0]
schema = 'ai_lab_' + hashlib.sha256(user.encode()).hexdigest()[:10]
# Uses the current catalog; change it to a catalog where you can CREATE SCHEMA.
namespace = f'`{catalog}`.`{schema}`'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {namespace}')
def table(name):
    assert name.replace('_', '').isalnum()
    return f'{namespace}.`{name}`'
print('Course namespace:', namespace)


**Check your result:** A catalog and per-user course schema are printed.


## Step 2 — Create records and quality rules

Retain the full invalid record with a reason, a useful engineering pattern for AI training ingestion.


In [ ]:
raw = spark.createDataFrame([(1, 24., 1), (2, None, 0), (3, -5., 1), (4, 60., 0)], 'id long, age double, label int')
checked = raw.withColumn('reason', F.when(F.col('age').isNull(), 'missing_age').when(F.col('age') < 0, 'negative_age'))
good = checked.filter('reason IS NULL').drop('reason')
bad = checked.filter('reason IS NOT NULL')
assert good.count() == 2 and bad.count() == 2
display(bad)


**Check your result:** Two rows are quarantined with reasons.


## Step 3 — Persist and upsert

Only a course-owned table is overwritten. MERGE updates existing keys and inserts new ones; unique source keys are required.


In [ ]:
good.write.format('delta').mode('overwrite').saveAsTable(table('quality_customers'))
bad.write.format('delta').mode('overwrite').saveAsTable(table('quality_rejects'))
updates = spark.createDataFrame([(1, 25., 1), (5, 45., 0)], good.schema)
updates.createOrReplaceTempView('ai_quality_updates')
spark.sql(f"""MERGE INTO {table('quality_customers')} t USING ai_quality_updates s
ON t.id = s.id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
assert spark.table(table('quality_customers')).count() == 3
display(spark.table(table('quality_customers')).orderBy('id'))


**Check your result:** Three valid customer rows exist; id 1 now has age 25.


## Step 4 — Inspect lineage

Delta history records table operations. Table version numbers change on reruns, so do not hardcode a version as the latest.


In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {table('quality_customers')}"))


**Check your result:** History contains overwrite and MERGE operations.


## Practice

Add a duplicate source id and implement a deterministic winner using event timestamp before MERGE.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Use a window partitioned by id, ordered by event timestamp descending plus a stable tie-breaker; keep row_number = 1. Do not randomly drop duplicates when updates disagree.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
